# Run a transient and set tolerances

[Get a steady solve to converge](get-a-steady-solve-to-converge.ipynb) ends at a solved operating point. This page runs a transient from one, the loop of [Primary Cooling System Coastdown](../PCSCoastdown.ipynb) after its pump stops: with the defaults, with each option against the analytic flow, with a sign constraint, and once to failure.

In [1]:
import numpy as np
import pandas as pd

from stream import create_constraints
from stream.aggregator import Solution
from stream.calculations import Inertia, Junction, KirchhoffWDerivatives, Pump
from stream.composition import FlowGraph, ResistorFromKnownPoint, flow_edge
from stream.jacobians import DAE_jacobian
from stream.solvers import TransientRuntimeError
from stream.substances import light_water
from stream.units import hour

## Start from the steady state

The loop is wired as in [Wire a flow network](wire-a-flow-network.ipynb), the flywheel in `inertial_comps`. Solve the steady state first and start from it: the differential rows of `y0`, here the edge flows, are where the transient starts, and IDA keeps them as given. By default (`compute_initcond="yp0"`) it makes only the algebraic rows and the derivatives consistent with them. `pump.p = 0.0` then stops the pump.

In [2]:
T, head, inertia = 20.0, 1.6e5, 8e3
mdot0 = 2000 / hour * float(light_water.density(T))
pump = Pump(pressure=head, name="Pump")
flywheel = Inertia(inertia=inertia, name="Flywheel")
pcs = ResistorFromKnownPoint(dp=-head, mdot=mdot0, behavior="parabolic", Tin=T, fluid=light_water, name="PCS")
a, b = Junction(name="A"), Junction(name="B")
fg = FlowGraph(
    flow_edge((a, b), pump, flywheel),
    flow_edge((b, a), pcs),
    funcs={pcs: {"Tin": T}},
    inertial_comps=[flywheel],
    k_constructor=KirchhoffWDerivatives,
)
agr, K = fg.aggregator, fg.kirchhoff
y0 = agr.solve_steady(fg.guess_steady_state({pump: mdot0, pcs: mdot0}, T))
row = agr.var_index(K, K.component_edge(pcs))
print(f"steady flow:       {y0[row]:.2f} kg/s")
print(f"residual:          {np.linalg.norm(agr.compute(y0, 0.0)):.1e}")
print(f"differential rows: {int(agr.mass.sum())} of {len(agr)}")
pump.p = 0.0

steady flow:       554.42 kg/s
residual:          2.9e-11
differential rows: 2 of 12


The residual is about 3e-11. `solve` picks its backend from the mass vector: ODE when every row is differential, DAE (IDA) when some are, a quasi-static solve when none are. Here two are, so IDA runs. `eq_type="DAE"`, or `"ODE"` or `"ALG"`, overrides it.

## Tolerances, a Jacobian and a step cap

Every option `solve` does not take itself goes to the backend, under the backend's own name. For IDA:

- `rtol` is one number for the whole system, `1e-6` by default.
- `atol` is a float or an array with one entry per row, `1e-12` by default. `agr.scaled_atol(rel)` builds the array: `rel` times each row's nominal magnitude in `stream.scales.DEFAULT_SCALES`, keyed by variable name, `1.0` for a name it lacks.
- `jacfn=DAE_jacobian(agr)` hands IDA a finite-difference Jacobian that, per column, re-evaluates only the calculation owning that variable and those reading it.
- `max_step_size` caps the internal step, `0.0` meaning no cap; `max_steps` caps the internal steps between two output times, `0` meaning IDA's 500.

A name IDA does not know, such as scipy's `max_step`, raises a `ValueError`. The cell runs the first 300 s four ways and compares the flow at 100 s with the analytic coastdown, $\dot m(t) = \dot m_0 / (1 + \alpha \dot m_0 t)$, which [the coastdown example](../PCSCoastdown.ipynb#Analytic-Solution-of-the-Transient) derives.

In [3]:
t = np.linspace(0.0, 300.0, 61)
alpha = abs(pcs.dp_out(mdot=1.0, Tin=T)) / inertia
exact = mdot0 / (1 + alpha * mdot0 * 100.0)
scaled = {"rtol": 1e-6, "atol": agr.scaled_atol(1e-6), "jacfn": DAE_jacobian(agr)}
options = {
    "defaults": {},
    "rtol 1e-3, scaled atol": {"rtol": 1e-3, "atol": agr.scaled_atol(1e-3)},
    "rtol 1e-6, scaled atol, jacfn": scaled,
    "as above, max_step_size 0.1": scaled | {"max_step_size": 0.1},
}
rows = {}
for label, opts in options.items():
    sol = agr.solve(y0, t, **opts)
    flow = sol.data[t == 100.0, row][0]
    rows[label] = {
        "flow at 100 s [kg/s]": f"{flow:.4f}",
        "relative error": f"{(flow - exact) / exact:.1e}",
    }
print(f"analytic flow at 100 s: {exact:.4f} kg/s")
print(pd.DataFrame(rows).T.to_string())
try:
    agr.solve(y0, t, max_step=0.1)
except ValueError as err:
    print(f"ValueError: {err}")

analytic flow at 100 s: 120.3329 kg/s
                              flow at 100 s [kg/s] relative error
defaults                                  120.3329       -5.5e-09
rtol 1e-3, scaled atol                    120.1961       -1.1e-03
rtol 1e-6, scaled atol, jacfn             120.3330        6.1e-07
as above, max_step_size 0.1               120.3331        1.3e-06
ValueError: Option 'max_step' is not supported by solver


The defaults land closest, a relative error under 1e-8. A relative `1e-3` misses by about 0.1 %, the scaled `1e-6` by about 6e-7. The step cap buys nothing here.

## Loops with events

Pass `exclude_algvar_from_error=True` on a loop where one edge's flow is pinned by an algebraic law, a flapper, while the rest carries inertia: it leaves the algebraic rows out of IDA's error test, which keeps the step from collapsing after an event restart at tight tolerance. Leave it off on an inertia-free network whose event margin reads an algebraic flow: the event is then localised late. `continuous` decides what an event's stop request does: `False`, the default, stops the run; `True` restarts past it. The cell makes the `stream.examples.lofa` call plus `continuous=True`; this loop has no events for either to act on.

In [4]:
lofa_style = agr.solve(y0, t, jacfn=DAE_jacobian(agr), atol=agr.scaled_atol(1e-4), rtol=1e-4,
                       max_steps=1_000_000, exclude_algvar_from_error=True, continuous=True)
flow = lofa_style.data[t == 100.0, row][0]
print(f"flow at 100 s:  {flow:.4f} kg/s")
print(f"relative error: {(flow - exact) / exact:.1e}")

flow at 100 s:  120.3483 kg/s
relative error: 1.3e-04


The run completes, about 1e-4 from the analytic flow, the size of its tolerances.

## Keep a variable on one side of zero

`create_constraints(agr, ...)` builds IDA's sign vector, one entry per row: each keyword is the name of a `CONSTRAINT` member (`negative`, `non_positive`, `none`, `non_negative`, `positive`) and takes a list of variable names; every other row takes `default_sign`, `CONSTRAINT.none` by default. Pass it as `constraints_type`, on the DAE backend only. The cell keeps the return flow non-negative.

In [5]:
signs = create_constraints(agr, non_negative=[K.component_edge(pcs)])
print(signs)
held = agr.solve(y0, t, constraints_type=signs, **scaled)
print(f"lowest return flow: {held.data[:, row].min():.2f} kg/s")
print(f"flow at 100 s:      {held.data[t == 100.0, row][0]:.4f} kg/s")

[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
lowest return flow: 46.90 kg/s
flow at 100 s:      120.3330 kg/s


The flow stays above zero and matches the scaled run: a coastdown never reaches zero, so the constraint has nothing to stop.

## When a run fails

A failed run raises `TransientRuntimeError`, whose `t` and `y` hold the trajectory up to the failure. Here a step cap of 1 s meets a 280 s output interval with `max_steps` at 50. `err_handler` takes IDA's own stderr message; the cell drops it.

In [6]:
try:
    agr.solve(y0, [0.0, 10.0, 20.0, 300.0], max_step_size=1.0, max_steps=50, err_handler=lambda *args: None)
except TransientRuntimeError as err:
    partial = Solution(err.t, err.y)
    print(err)
    print(f"times kept: {partial.time}")
    print(f"flow at 20 s: {partial.data[-1, row]:.2f} kg/s")

At t = 20.00000: Could not reach endpoint
times kept: [ 0. 10. 20.]
flow at 20 s: 322.06 kg/s


The trajectory holds the output times reached, up to 20 s. [Diagnose a failing solve](diagnose-a-failing-solve.ipynb) takes it from there.

[Both Pumps Trip](../LOFABothPumpsTrip.ipynb) runs an hour-long transient with these options, and [How far to trust the event times](../LOFAVariations.ipynb#How-far-to-trust-the-event-times) reruns its first five minutes at two tolerances. The usual next task: [Trip, stop and restart](trip-stop-and-restart.ipynb).